# School Attendance Data Exploration and Cleaning

This notebook explores the Connecticut school attendance dataset, performs data cleaning, normalization, clustering, and generates visualizations, including 3D plots of the resulting clusters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA

sns.set(style='whitegrid', context='notebook')
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')  # suppress warnings

In [ ]:
df = pd.read_csv('School_Attendance_dataset.csv')

# Inspect head and info
df.head(), df.info()

In [ ]:
# Visualize missing values
missing_counts = df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x=missing_counts.index, y=missing_counts.values, palette='coolwarm')
plt.title('Missing values by column')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Number of missing values')
plt.tight_layout()
plt.show()

missing_counts

In [ ]:
# Remove Connecticut State and All Students rows
df = df[~df['District code'].str.startswith('0')].copy()
df = df[~df['Student group'].str.contains('All Students')].copy()


In [ ]:
# Count Duplicate Rows
duplicate_rows = df[df.duplicated()]
print(f"Number of duplicate rows: {len(duplicate_rows)}")

In [ ]:
# Helper functions for cleaning and clustering

# Impute missing values
def impute_numerical(df, numeric_cols):
    for col in numeric_cols:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
    return df

# Impute categorical values
def impute_categorical(df, categorical_cols):
    for col in categorical_cols:
        if df[col].isnull().all():
            df[col] = df[col].fillna('Unknown')
        else:
            mode_val = df[col].mode().iloc[0]
            df[col] = df[col].fillna(mode_val)
    return df

# Cap outliers
def cap_outliers(df, numeric_cols):
    for col in numeric_cols:
        lower = df[col].quantile(0.01)
        upper = df[col].quantile(0.99)
        df[col] = np.where(df[col] < lower, lower, df[col])
        df[col] = np.where(df[col] > upper, upper, df[col])
    return df

# Standardize features (Scaling)
def standardize_features(df, numeric_cols):
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df[numeric_cols])
    scaled_df = pd.DataFrame(
        scaled_data,
        columns=[f"{col}_scaled" for col in numeric_cols],
        index=df.index,
    )
    return scaled_df

# Determine optimal number of clusters
def determine_optimal_clusters(X, cluster_range=range(2, 6)):
    best_k = 2
    best_score = -1
    for k in cluster_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        score = silhouette_score(X, labels)
        print(f"Silhouette score for k={k}: {score:.4f}")
        if score > best_score:
            best_k = k
            best_score = score
    return best_k

# Apply KMeans clustering
def apply_kmeans(df, numeric_cols):
    scaled_df = standardize_features(df, numeric_cols)
    optimal_k = determine_optimal_clusters(scaled_df.values)
    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(scaled_df.values)
    df['Cluster'] = cluster_labels
    for col in scaled_df.columns:
        df[col] = scaled_df[col]
    return df



In [ ]:
# Define columns
numeric_cols = [
    '2021-2022 student count - year to date',
    '2021-2022 attendance rate - year to date',
    '2020-2021 student count',
    '2020-2021 attendance rate',
    '2019-2020 student count',
    '2019-2020 attendance rate'
]
categorical_cols = ['Category', 'Student group']

clean_df = df.copy()
clean_df = impute_numerical(clean_df, numeric_cols)
clean_df = impute_categorical(clean_df, categorical_cols)
clean_df = clean_df.drop_duplicates()
clean_df = cap_outliers(clean_df, numeric_cols)
clean_df = apply_kmeans(clean_df, numeric_cols)

# Show cleaned data summary
clean_df[['District code', 'District name', 'Cluster'] + numeric_cols].head()

In [ ]:
# Cluster summary statistics
cluster_summary = clean_df.groupby('Cluster')[[
    '2021-2022 attendance rate - year to date',
    '2020-2021 attendance rate',
    '2019-2020 attendance rate'
]].mean()

cluster_counts = clean_df['Cluster'].value_counts()

print('Cluster mean attendance rates:')
print(cluster_summary)
print('Cluster counts:')
print(cluster_counts)

cluster_summary, cluster_counts

In [ ]:
# 3D scatter plot of attendance rates (raw values)
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

x = clean_df['2019-2020 attendance rate']
y = clean_df['2020-2021 attendance rate']
z = clean_df['2021-2022 attendance rate - year to date']
clusters = clean_df['Cluster']

scatter = ax.scatter(x, y, z, c=clusters, cmap='viridis', s=20)
ax.set_xlabel('2019-2020 attendance rate')
ax.set_ylabel('2020-2021 attendance rate')
ax.set_zlabel('2021-2022 attendance rate')
ax.set_title('3D Cluster Visualization (Raw Attendance Rates)')
legend1 = ax.legend(*scatter.legend_elements(), title='Cluster')
ax.add_artist(legend1)
plt.show()

Reduction of Data used to aid in better presentation of information.

In [ ]:
# --- Attendance-only clustering ---
attendance_cols = [
    '2019-2020 attendance rate',
    '2020-2021 attendance rate',
    '2021-2022 attendance rate - year to date'
]

# Standardize attendance features
attendance_scaled = StandardScaler().fit_transform(clean_df[attendance_cols])

# Determine the optimal number of clusters (k) based on silhouette score
best_k_att = 2
best_score_att = -1
for k in range(2, 6):
    kmeans_att = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_att = kmeans_att.fit_predict(attendance_scaled)
    score_att = silhouette_score(attendance_scaled, labels_att)
    print(f"Attendance-only silhouette score for k={k}: {score_att:.4f}")
    if score_att > best_score_att:
        best_k_att = k
        best_score_att = score_att

# Fit final KMeans model with optimal k
kmeans_att_final = KMeans(n_clusters=best_k_att, random_state=42, n_init=10)
attendance_clusters = kmeans_att_final.fit_predict(attendance_scaled)
clean_df['AttendanceCluster'] = attendance_clusters

# Calculate cluster-level mean attendance to assign descriptive labels
attendance_summary = clean_df.groupby('AttendanceCluster')[attendance_cols].mean()
attendance_summary['MeanAttendance'] = attendance_summary.mean(axis=1)

# Rank clusters by mean attendance and assign labels
sorted_clusters = attendance_summary['MeanAttendance'].sort_values().index.tolist()
cluster_labels_map = {}
for i, cid in enumerate(sorted_clusters):
    if i == 0:
        cluster_labels_map[cid] = 'Low Attendance (Needs Intervention)'
    elif i == len(sorted_clusters) - 1:
        cluster_labels_map[cid] = 'High Attendance (No Intervention)'
    else:
        cluster_labels_map[cid] = 'Moderate Attendance'

clean_df['AttendanceClusterLabel'] = clean_df['AttendanceCluster'].map(cluster_labels_map)

# Display unweighted and weighted mean attendance rates for each cluster
weights_att = clean_df[['2019-2020 student count', '2020-2021 student count', '2021-2022 student count - year to date']].mean(axis=1)
cluster_means_unweighted = clean_df.groupby('AttendanceCluster')[attendance_cols].mean()
cluster_means_weighted = clean_df.groupby('AttendanceCluster').apply(lambda g: (g[attendance_cols].multiply(g[['2019-2020 student count', '2020-2021 student count', '2021-2022 student count - year to date']].mean(axis=1), axis=0)).sum() / g[['2019-2020 student count', '2020-2021 student count', '2021-2022 student count - year to date']].mean(axis=1).sum())

summary_df = pd.concat({
    'Unweighted Mean': cluster_means_unweighted,
    'Weighted Mean': cluster_means_weighted
}, axis=1)
print('Attendance-only cluster summary (unweighted and weighted means):')
display(summary_df)

# Display distribution of student groups across the attendance clusters
group_distribution = clean_df.groupby(['AttendanceClusterLabel', 'Student group']).size().unstack(fill_value=0)
group_distribution_pct = group_distribution.div(group_distribution.sum(axis=0), axis=1) * 100
print('')
print('Student group distribution by attendance cluster (counts):')
display(group_distribution)
print('')
print('Student group distribution by attendance cluster (percentages):')
display(group_distribution_pct)

# Display distribution of student counts across the attendance clusters
student_count_distribution = clean_df.groupby(['AttendanceClusterLabel', 'Student group'])[['2019-2020 student count', '2020-2021 student count', '2021-2022 student count - year to date']].sum()
print('')
print('Student count distribution by attendance cluster (Sums):')
display(student_count_distribution)

In [ ]:
# Pairplot and 3D visualization for attendance-only clusters

## Pairplot of attendance rates colored by descriptive cluster label
sns.pairplot(
    clean_df[attendance_cols + ['AttendanceClusterLabel']],
    hue='AttendanceClusterLabel',
    diag_kind='hist',
    corner=True,
    palette='viridis',
    size=4
)
plt.suptitle('Pairwise Scatter Plots of Attendance Rates by Cluster Label', y=1.0, fontsize=10)
plt.show()

## 3D scatter plot of raw attendance rates colored by descriptive cluster label
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')
x = clean_df['2019-2020 attendance rate']
y = clean_df['2020-2021 attendance rate']
z = clean_df['2021-2022 attendance rate - year to date']
labels = clean_df['AttendanceClusterLabel']
scatter = ax.scatter(x, y, z, c=clean_df['AttendanceCluster'], cmap='viridis', s=20)
ax.set_xlabel('2019-2020 Attendance Rate')
ax.set_ylabel('2020-2021 Attendance Rate')
ax.set_zlabel('2021-2022 Attendance Rate')
ax.set_title('3D Attendance Cluster Visualization (Raw Rates)')
legend1 = ax.legend(*scatter.legend_elements(prop="colors"), title="Attendance Cluster Label")
ax.add_artist(legend1)
plt.show()